**Unidade curricular:** Visualização de Dados 2026/27 (FCUL)

**Grupo VD12:** Diva Manuel · Lêrice Germano · Matilde Quitério


## 1. Tema e dataset

O dataset escolhido para este projeto foi o **Health Indicators Dataset for Forecasting (WHO)**, um dataset que contém dados reais, usado para análise de indicadores de saúde à escala mundial. Contém mais de 200 países, 24 anos de indicadores de saúde, e mais de 50 indicadores. Disponível no kaggle.

**Link:** https://www.kaggle.com/datasets/patelris/who-health-dataset-43-indicators-200-countries 

**Fonte:** A fonte primária de referencia é o World Bank Open Data API. O kaggle é um intermediário que reune e reorganiza a informação e a fornece em datasets. 

**O conjunto de dados está estruturado em três ficheiros:**
- `health_indicators.csv`: Painel temporal com 48 variáveis, 211 países e territórios, cobrindo o período de 2000 a 2024 (5 275 linhas de observações), onde cada linha representa um país num ano específico, e as colunas os indicadores de saúde. Este é o dataset principal para a nossa análise.

- `country_latest.csv` : Tabela com 8 colunas. Registo anual mais recente disponível por país e por indicador. 
- `indicator_metadata.csv`: Metadados descritivos com o código, unidade e categoria de cada indicador.


## 2. Objetivo Geral
Construir uma aplicação de visual analytics que permita explorar como evoluiu a esperança média de vida desde 2000, em que medida a riqueza e o investimento em saúde a explicam, que países fogem à regra e que perfís de países existem. 



### 2.1. Perguntas analíticas
**Pergunta 1:** Como evoluiu a esperança média de vida a nível global entre 2000 e 2024, e que padrões de desigualdade persistem entre regiões geográficas, grupos de rendimento e géneros?

**Pergunta 2:** Em que medida estão o desenvolvimento económico e o investimento em saúde associados à esperança média de vida, e que países apresentam resultados de saúde acima ou abaixo do esperado face aos recursos?

**Pergunta 3:** Qual é o padrão geográfico da mortalidade e que grupos de países apresentam comportamentos semelhantes com base nos seus indicadores de saúde e mortalidade? 


## 3.  Carregamento de dados

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
plt.style.use("seaborn-v0_8-whitegrid")

In [ ]:
# Carregamento dos ficheiros
df_health_indicators = pd.read_csv("health_indicators.csv")
df_meta = pd.read_csv("indicator_metadata.csv")
df_latest = pd.read_csv("country_latest.csv")


Shape do dataset de indicadores: (5275, 48)
Países analisados: 211


### 3.1 Exploração e diagnóstico inicial

#### 3.1.1 Visão geral

In [18]:
print("Shape:", df_health_indicators.shape)
print("Anos:", df_health_indicators["year"].min(), "a", df_health_indicators["year"].max())
print("Países/territórios:", df_health_indicators["country_name"].nunique())
print("Colunas:", df_health_indicators.columns.tolist())

display(df_health_indicators.head())
display(df_health_indicators.dtypes.to_frame("Tipos de dados"))

Shape: (5275, 48)
Anos: 2000 a 2024
Países/territórios: 211
Colunas: ['country_code', 'country_name', 'region', 'income_level', 'year', 'population', 'population_growth', 'urban_pct', 'life_expectancy', 'life_expectancy_male', 'life_expectancy_female', 'death_rate', 'birth_rate', 'health_expenditure_pct_gdp', 'health_expenditure_per_capita', 'physicians_per_1000', 'hospital_beds_per_1000', 'nurses_per_1000', 'under5_mortality', 'neonatal_mortality', 'maternal_mortality', 'tb_incidence', 'hiv_incidence', 'communicable_death_pct', 'noncommunicable_death_pct', 'smoking_male', 'smoking_female', 'obesity_pct', 'alcohol_per_capita', 'diabetes_pct', 'safe_water_pct', 'sanitation_pct', 'handwashing_pct', 'immunization_measles', 'immunization_dpt', 'immunization_bcg', 'immunization_pol3', 'immunization_hib3', 'stunting_pct', 'wasting_pct', 'overweight_children_pct', 'undernourishment_pct', 'fertility_rate', 'adolescent_fertility', 'prenatal_care_pct', 'gdp_per_capita', 'gdp_per_capita_ppp', 'po

,country_code,country_name,region,income_level,year,population,population_growth,urban_pct,life_expectancy,life_expectancy_male,...,stunting_pct,wasting_pct,overweight_children_pct,undernourishment_pct,fertility_rate,adolescent_fertility,prenatal_care_pct,gdp_per_capita,gdp_per_capita_ppp,poverty_rate
0,ABW,Aruba,Latin America & Caribbean,High income,2000,90588.0,1.030817,65.354550,72.939,70.149,...,NaN,NaN,NaN,NaN,1.845,43.729,NaN,20681.023027,30245.706974,NaN
1,ABW,Aruba,Latin America & Caribbean,High income,2001,91439.0,0.935033,65.335114,73.044,70.251,...,NaN,NaN,NaN,NaN,1.813,40.463,NaN,20740.132583,31920.239073,NaN
2,ABW,Aruba,Latin America & Caribbean,High income,2002,92074.0,0.692052,65.282069,73.135,70.364,...,NaN,NaN,NaN,NaN,1.800,38.185,NaN,21307.248251,31888.508737,NaN
3,ABW,Aruba,Latin America & Caribbean,High income,2003,93128.0,1.138229,65.198292,73.236,70.488,...,NaN,NaN,NaN,NaN,1.808,37.807,NaN,21949.485996,32507.084315,NaN
4,ABW,Aruba,Latin America & Caribbean,High income,2004,95138.0,2.135358,65.088653,73.223,70.487,...,NaN,NaN,NaN,NaN,1.819,38.761,NaN,23700.631990,35059.273098,NaN


,Tipos de dados
country_code,object
country_name,object
region,object
income_level,object
year,int64
population,float64
population_growth,float64
urban_pct,float64
life_expectancy,float64
life_expectancy_male,float64


In [19]:
#Estatísticas descritivas de todas as variáveis
display(df_health_indicators.describe(include="all"))

,country_code,country_name,region,income_level,year,population,population_growth,urban_pct,life_expectancy,life_expectancy_male,...,stunting_pct,wasting_pct,overweight_children_pct,undernourishment_pct,fertility_rate,adolescent_fertility,prenatal_care_pct,gdp_per_capita,gdp_per_capita_ppp,poverty_rate
count,5275,5275,5275,5275,5275.000000,5.275000e+03,5274.000000,5275.000000,5275.000000,5275.000000,...,844.000000,843.000000,806.000000,3762.000000,5275.000000,5275.000000,741.000000,5110.000000,4838.000000,1827.000000
unique,211,211,7,5,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
top,ABW,Aruba,Europe & Central Asia,High income,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
freq,25,25,1400,2025,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
mean,NaN,NaN,NaN,NaN,2012.000000,3.378305e+07,1.280475,58.716732,70.748244,68.215760,...,24.009360,5.896679,6.198015,10.937400,2.797145,51.097317,86.791826,15749.091058,19781.847712,9.492118
std,NaN,NaN,NaN,NaN,7.211786,1.323408e+08,1.618914,23.674753,8.701547,8.471691,...,15.043393,4.401812,4.499476,11.043652,1.468998,42.893201,16.137098,25074.200336,22391.204701,17.142033
min,NaN,NaN,NaN,NaN,2000.000000,9.544000e+03,-11.356645,8.043814,14.665000,12.383000,...,0.500000,0.100000,0.000000,2.500000,0.721000,0.505000,16.100000,109.593814,403.983197,0.000000
25%,NaN,NaN,NaN,NaN,2006.000000,7.743870e+05,0.297465,39.060949,65.356000,62.722000,...,10.800000,2.400000,2.725000,2.500000,1.670000,15.427500,82.500000,1544.567936,3899.860735,0.300000
50%,NaN,NaN,NaN,NaN,2012.000000,5.943366e+06,1.175030,60.031069,72.442000,69.217000,...,23.800000,4.900000,5.300000,6.300000,2.274000,39.131000,92.800000,5356.365656,11309.514697,1.500000
75%,NaN,NaN,NaN,NaN,2018.000000,2.201484e+07,2.210221,77.627285,77.197000,74.500000,...,35.325000,8.700000,8.700000,15.600000,3.638500,75.234000,97.600000,20032.321089,28679.803239,9.800000


#### 3.1.2 Analise de variáveis categóricas

In [4]:
df_health_indicators.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5275 entries, 0 to 5274
Data columns (total 48 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   country_code                   5275 non-null   object 
 1   country_name                   5275 non-null   object 
 2   region                         5275 non-null   object 
 3   income_level                   5275 non-null   object 
 4   year                           5275 non-null   int64  
 5   population                     5275 non-null   float64
 6   population_growth              5274 non-null   float64
 7   urban_pct                      5275 non-null   float64
 8   life_expectancy                5275 non-null   float64
 9   life_expectancy_male           5275 non-null   float64
 10  life_expectancy_female         5275 non-null   float64
 11  death_rate                     5275 non-null   float64
 12  birth_rate                     5275 non-null   f

In [16]:
display(df_health_indicators.select_dtypes(include=['object']))


,country_code,country_name,region,income_level
0,ABW,Aruba,Latin America & Caribbean,High income
1,ABW,Aruba,Latin America & Caribbean,High income
2,ABW,Aruba,Latin America & Caribbean,High income
3,ABW,Aruba,Latin America & Caribbean,High income
4,ABW,Aruba,Latin America & Caribbean,High income
...,...,...,...,...
5270,ZWE,Zimbabwe,Sub-Saharan Africa,Lower middle income
5271,ZWE,Zimbabwe,Sub-Saharan Africa,Lower middle income
5272,ZWE,Zimbabwe,Sub-Saharan Africa,Lower middle income
5273,ZWE,Zimbabwe,Sub-Saharan Africa,Lower middle income


In [7]:
print("Regiões presentes:\n", df_health_indicators['region'].unique())

print("\nGrupos de rendimento:\n", df_health_indicators['income_level'].unique())

Regiões presentes:
 ['Latin America & Caribbean '
 'Middle East, North Africa, Afghanistan & Pakistan' 'Sub-Saharan Africa '
 'Europe & Central Asia' 'East Asia & Pacific' 'South Asia'
 'North America']

Grupos de rendimento:
 ['High income' 'Low income' 'Lower middle income' 'Upper middle income'
 'Not classified']


##### Observação sobre as variáveis categóricas
- **`region`**: Existem 7 regiões globais representadas
- **`income_level`**: Os países estão divididos em 5 patamares de riqueza (*Low income*, *Lower middle income*, *Upper middle income*, *High income* and *Not classified*)

- `country_code`, `country_name`,`region`, `income_level` são interpretados como texto genérico *(object)*. É possivel existirem espaços em branco acidentais no inicio ou fim das palavras em texto genérico. Essa existencia de espaços residuias provoca erros silenciosos em operações de filtragem e pesquisa, por isso será necessário fazer um tratamento nesta variáveis para garantir que os nomes fiquem padronizados e pesquisáveis sem inconsistência

- A classificação económica dos países *(income_level)*, por estar como object não tem uma hierarquia definida no dataset. Será necessário tornar esta variável numa variável categórica ordinal para assegurar que todos os gráficos e agregações respeitem essa progressão socioeconómica.

#### 3.1.3 Analise de duplicados


In [4]:
#Verificação de registos duplicados por país/ano
duplicados = df_health_indicators.duplicated(subset=['country_name', 'year']).sum()
print(f"Registos duplicados por país/ano: {duplicados}")

Registos duplicados por país/ano: 0


O resultado de registos duplicados foi 0, o que garante que a combinação (country_name, year) é única, ou seja, não existe nenhum país repetido no mesmo ano.

#### 3.1.4 Analise de missing values

In [6]:
#Diagnóstico de valores nulos (missing values) nas variáveis dos temas das nossas perguntas analíticas

colunas_analise = ['life_expectancy', 'life_expectancy_male','life_expectancy_female','gdp_per_capita', 'health_expenditure_per_capita', 'health_expenditure_pct_gdp', 'death_rate']

#construção da tabela de diagnóstico de valores nulos
tabela_nulos = pd.DataFrame({
    "Total_registos": len(df_health_indicators),
    "Valores_nulos": df_health_indicators[colunas_analise].isnull().sum(),
    "Percentagem_nulos": df_health_indicators[colunas_analise].isnull().mean() * 100}).round(2)

tabela_nulos 


,Total_registos,Valores_nulos,Percentagem_nulos
life_expectancy,5275,0,0.00
life_expectancy_male,5275,0,0.00
life_expectancy_female,5275,0,0.00
gdp_per_capita,5275,165,3.13
health_expenditure_per_capita,5275,764,14.48
health_expenditure_pct_gdp,5275,761,14.43
death_rate,5275,0,0.00


Variáveis como life_expectancy,life_expectancy_male,life_expectancy_female, death_rate têm zero valores em falta. Isto significa que as análises que envolvam estas variáveis possam ser feita sem perda de amostra ou sem necessidade de interpolação.

As variáveis económicas e de despesa em saúde ja apresentam uma certa percentagem de missing values, o que pode afetar algum tratamento das questões analíticas: 

- gdp_per_capita apresenta 3.13% de valores em falta (165 observações em falta)
- health_expenditure_per_capita apresenta 14.48% de valores em falta (764 observações em falta)
- health_expenditure_pct_gdp apresenta 14.43% de valores em falta (761 observações em falta)

Para interpretar com rigor metodológico a causa destes nulos, desenvolvemos alguns testes por **ano** e por **pais**





In [ ]:
nulos_anuais = df_health_indicators.groupby('year')[['gdp_per_capita','health_expenditure_per_capita','health_expenditure_pct_gdp']].apply(lambda x: x.isnull().sum())

print("Evolução dos nulos por ano: ")
nulos_anuais


Evolução dos nulos por ano: 


,gdp_per_capita,health_expenditure_per_capita,health_expenditure_pct_gdp
year,,,
2000,12,29,29
2001,11,29,29
2002,7,28,28
2003,7,26,26
2004,7,26,26
2005,7,26,26
2006,6,26,26
2007,6,26,26
2008,4,26,26


- De 2000 até 2023, o número de paises sem dados de despesa em saúde manteve-se estável, oscilando entre 20 e 29. Em 2024 esse valor salta abruptamente para 189 países sem dados de despesas de saúde, ou seja 86.6% do dataset não tem dados financeiros em 2024. Isto faz com que qualquer gráfico ou modelo que use deapesas (health_expenditure) deve focar-se na janela 2000 a 2022/2023 para evitar distorções.

- O Pib per capita (`gdp_per_capita`) tem apenas entre só entre 2 a 7 paises em falta por ano, havendo uma ligeira subida em 2023 (13 nulos) e 2024 (23 nulos).

## 4.  Limpeza e Transformação de dados


In [23]:
#criação da cópia do dataset para preservar o original e isolar transformações e limpeza de dados
dados = df_health_indicators.copy()

# Limpeza simples: retirar espaços a mais no texto nas variáveis interpretadas como object

dados['region'] = dados['region'].str.strip()
dados['income_level'] = dados['income_level'].str.strip()
dados['country_name'] = dados['country_name'].str.strip()
dados['country_code'] = dados['country_code'].str.strip()

# Definir a ordem lógica dos grupos de rendimento (do menor para o maior) para efeitos de visualização
ordem_rendimento = ['Low income', 'Lower middle income', 'Upper middle income', 'High income']
dados['income_level'] = pd.Categorical(dados['income_level'], categories=ordem_rendimento, ordered=True)


# 5. Exploração Visual dos Dados

### Pergunta 1: Como evoluiu a esperança média de vida a nível global entre 2000 e 2024, e que padrões de desigualdade persistem entre regiões geográficas, grupos de rendimento e géneros?


### Pergunta 2: Em que medida estão o desenvolvimento económico e o investimento em saúde associados à esperança média de vida, e que países apresentam resultados de saúde acima ou abaixo do esperado face aos recursos?

De que modo o PIB per capita influencia a esperança média de vida e a partir de que patamar se observam retornos decrescentes?

O gasto em saúde (per capita e % do PIB) está associado a ganhos de esperança de vida para além do efeito do PIB?

Que países têm esperança de vida elevada com poucos recursos económicos, e quais têm resultados baixos apesar de gastarem muito?